# **1. Loading Dataset**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the dataset
df = pd.read_csv("telco_churn.csv")

# 2. Show the dataset as soon as it loads
df.head()

# **2. Check First and Last Rows**

In [ ]:
# To check first 5 rows:
df.head()

# To check last 5 rows:
df.tail()

# **3. Check Info, Missing Values and Duplicates**

In [ ]:
# To check the data types:
df.info()

# To check missing values:
print(df.isnull().sum())

# To check the duplicate rows:
print("Duplicates:", df.duplicated().sum())

# To check the target column:
print(df["Churn"].value_counts())
print("Churn rate:", round((df["Churn"] == "Yes").mean() * 100, 2), "%")

# **4. Clean the Data**

In [ ]:
# 1. Drop the customerID column (it is just an ID, not a feature)
df = df.drop("customerID", axis=1)

# 2. Convert TotalCharges to numeric (11 values are blank spaces, they become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 3. Drop rows where TotalCharges is missing (only 11 rows)
df = df.dropna()

# 4. Remove duplicate rows
df = df.drop_duplicates()

# 5. Convert target to binary: Yes = 1 (churned), No = 0 (stayed)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print("Shape after cleaning:", df.shape)
print(df["Churn"].value_counts())
print("Churn rate:", round(df["Churn"].mean() * 100, 2), "%")

# **5. EDA — Graphs**

In [ ]:
# 1. Churn count plot
sns.countplot(x="Churn", data=df)
plt.title("Customer Churn (0 = Stayed, 1 = Churned)")
plt.show()

# 2. Churn rate by contract type
print(df.groupby("Contract")["Churn"].mean().round(3))
sns.barplot(x="Contract", y="Churn", data=df)
plt.title("Churn Rate by Contract Type")
plt.show()

# 3. Tenure vs churn
sns.boxplot(x="Churn", y="tenure", data=df)
plt.title("Tenure vs Churn")
plt.show()

# 4. Monthly charges vs churn
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

# 5. Churn rate by internet service
print(df.groupby("InternetService")["Churn"].mean().round(3))

# 6. Churn rate by payment method
print(df.groupby("PaymentMethod")["Churn"].mean().round(3))

# **6. Prepare and Split the Data**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separate features (X) and target (y)
X = df.drop("Churn", axis=1)
y = df["Churn"]

# 2. One-hot encode the categorical columns
X = pd.get_dummies(X, drop_first=True)

# 3. Remember the feature columns (the app needs them in the same order)
feature_columns = X.columns.tolist()

# 4. Split into train (80%) and test (20%), stratified so churn rate stays the same
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Scale only the numeric columns
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Number of features:", len(feature_columns))

# **7. Train the Models**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

# **8. Evaluate the Models**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results)
print(results_df)

# Best model = highest ROC-AUC
best_name = results_df.loc[results_df["ROC-AUC"].idxmax(), "Model"]
best_model = models[best_name]
print("\nBest model:", best_name)

# Confusion matrix and report for the best model
print(confusion_matrix(y_test, best_model.predict(X_test)))
print(classification_report(y_test, best_model.predict(X_test)))

# **9. Save the Best Model and Predict a New Customer**

In [ ]:
import joblib

# Save the best model
joblib.dump(best_model, "churn_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(feature_columns, "feature_columns.joblib")
joblib.dump(numeric_cols, "numeric_cols.joblib")
print("Saved: churn_model.joblib, scaler.joblib, feature_columns.joblib, numeric_cols.joblib")

# Predict a new customer (values taken from a real row in the dataset)
new_customer = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
    "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service",
    "InternetService": "DSL", "OnlineSecurity": "No", "OnlineBackup": "Yes",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Month-to-month",
    "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 29.85, "TotalCharges": 29.85,
}])
new_customer = pd.get_dummies(new_customer)
new_customer = new_customer.reindex(columns=feature_columns, fill_value=0)
new_customer[numeric_cols] = scaler.transform(new_customer[numeric_cols])
risk = best_model.predict_proba(new_customer)[0][1] * 100
print("Prediction:", best_model.predict(new_customer)[0], "| Churn Risk:", round(risk, 1), "%")